# CDSSwarm - Download ERA5 in Parallel! 
Download ERA5 data in parallel using `cdsswarm`. [Readme - setup instructions](https://github.com/bgiebl/cdsswarm/blob/a99168512aecb04e1ca418925fd43c5cbc38d926/README.md)

In [1]:
import cdsswarm

In [6]:
# Define the number of workers for parallel downloading.
NUM_WORKERS = 16

In [ ]:
# CREATE TASK TEMPLATES FOR EACH JOB (SFC and PL) - we will expand these by year and month to parallise by year/month.
task_templates = [
    cdsswarm.Task(
        dataset="reanalysis-era5-single-levels",
        request={
            "product_type": ["reanalysis"],
            "variable": ["2m_temperature", "165", "166", "total_precipitation"], # Works with codes and names.
            "levtype": ["pl"],
            "year": ["2024", "2025"],
            "month": ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"],
            "day": ["01", "02", "03"],
            "time": ["12:00"],
            "data_format": "grib",
        },
        target="era5_sfc.grib",
    ),
    cdsswarm.Task(
        dataset="reanalysis-era5-pressure-levels",
        request={
            "product_type": ["reanalysis"],
            "variable": ["130", "131", "132" ],
            "levtype": ["pl"],
            "levellist": ["1000", "850", "500", "100"],
            "year": ["2024", "2025"],
            "month": ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"],
            "day": ["01", "02", "03"],
            "time": ["12:00"],
            "data_format": "grib",
        },
        target="era5_pl.grib",
    ),
]

In [ ]:
# EXPAND THE TASKS OUT BY YEAR/MONTH.
# This function takes the task templates and creates individual tasks for each year/month combination, allowing for parallel downloads.

def expand_tasks_by_year_month(templates):
    """Expand task templates into individual tasks per year/month combination."""
    expanded_tasks = []
    output_dir = "output"
    for template in templates:
        years = template.request.get("year", [])
        months = template.request.get("month", [])
        base_target = template.target.rsplit(".", 1)
        
        for year in years:
            for month in months:
                # Create a new request dict with single year and month
                new_request = template.request.copy()
                new_request["year"] = [year]
                new_request["month"] = [month]
                
                # Generate unique target filename
                if len(base_target) == 2:
                    new_target = f"{output_dir}/{year}_{month}_{base_target[0]}.{base_target[1]}"
                else:
                    new_target = f"{output_dir}/{year}_{month}_{template.target}"
                
                expanded_tasks.append(cdsswarm.Task(
                    dataset=template.dataset,
                    request=new_request,
                    target=new_target,
                ))
    return expanded_tasks

# EXPAND THE TASKS OUT BY YEAR/MONTH.
tasks = expand_tasks_by_year_month(task_templates)
print(f"Tasks expanded to {len(tasks)} tasks.")

Tasks expanded to 48 tasks.


In [ ]:
# USE THE CDSSWARM PACKAGE TO DOWNLOAD THE TASKS IN PARALLEL
results = cdsswarm.download(tasks, num_workers=NUM_WORKERS)

Checking for reusable CDS jobs...
Found 48 reusable job(s)
  [Worker 11] starting 2024_12_era5_sfc.grib
  [Worker 9] starting 2024_10_era5_sfc.grib
  [Worker 7] starting 2024_08_era5_sfc.grib
  [Worker 4] starting 2024_05_era5_sfc.grib
  [Worker 10] starting 2024_11_era5_sfc.grib
  [Worker 13] starting 2025_02_era5_sfc.grib
  [Worker 1] starting 2024_02_era5_sfc.grib
  [Worker 3] starting 2024_04_era5_sfc.grib
  [Worker 5] starting 2024_06_era5_sfc.grib
  [Worker 8] starting 2024_09_era5_sfc.grib
  [Worker 12] starting 2025_01_era5_sfc.grib
  [Worker 2] starting 2024_03_era5_sfc.grib
  [Worker 6] starting 2024_07_era5_sfc.grib
  [Worker 14] starting 2025_03_era5_sfc.grib
  [Worker 15] starting 2025_04_era5_sfc.grib
  [Worker 0] starting 2024_01_era5_sfc.grib
  [Worker 13] 2025_02_era5_sfc.grib: successful — server finished, starting download
  [Worker 4] 2024_05_era5_sfc.grib: successful — server finished, starting download
  [Worker 9] 2024_10_era5_sfc.grib: successful — server finish

4fb04fb7bdee6fc75171d04cbf506b73.grib:   0%|          | 0.00/23.8M [00:00<?, ?B/s]

2f2d0da59362013b9edb8ec7a7d8dd82.grib:   0%|          | 0.00/23.8M [00:00<?, ?B/s]

992646fc4658ce5a779c41c5e3d5c0b.grib:   0%|          | 0.00/23.8M [00:00<?, ?B/s]

ee185d5071e381667eca10cf293f4211.grib:   0%|          | 0.00/23.8M [00:00<?, ?B/s]

...

  [Worker 8] 2025_11_era5_pl.grib: successful — server finished, starting download
  [Worker 2] 2025_05_era5_pl.grib: downloading 36/71 MB (50%)
  [Worker 6] 2024_12_era5_pl.grib: downloading 54/71 MB (75%)
  [Worker 0] 2025_03_era5_pl.grib: downloading 71/71 MB (100%)
  [32/48] 2025_03_era5_pl.grib done
  [Worker 7] 2025_07_era5_pl.grib: downloading 36/71 MB (50%)


6d08cdf0da427829ff3ca5dc78b7df56.grib:   0%|          | 0.00/71.3M [00:00<?, ?B/s]

  [Worker 3] 2024_09_era5_pl.grib: downloading 71/71 MB (100%)
  [33/48] 2024_09_era5_pl.grib done
  [Worker 0] starting 2025_12_era5_pl.grib
  [Worker 0] 2025_12_era5_pl.grib: successful — server finished, starting download
  [Worker 11] 2024_07_era5_pl.grib: downloading 36/71 MB (50%)
  [Worker 2] 2025_05_era5_pl.grib: downloading 54/71 MB (75%)
  [Worker 1] 2024_10_era5_pl.grib: downloading 71/71 MB (100%)
  [34/48] 2024_10_era5_pl.grib done
  [Worker 15] 2025_04_era5_pl.grib: downloading 18/71 MB (25%)
  [Worker 9] 2025_08_era5_pl.grib: downloading 18/71 MB (25%)
  [Worker 7] 2025_07_era5_pl.grib: downloading 54/71 MB (75%)


537a9b67fb9c92536d08d54346b948a3.grib:   0%|          | 0.00/71.3M [00:00<?, ?B/s]

  [Worker 6] 2024_12_era5_pl.grib: downloading 71/71 MB (100%)
  [35/48] 2024_12_era5_pl.grib done
  [Worker 14] 2025_06_era5_pl.grib: downloading 36/71 MB (50%)
  [Worker 5] 2024_11_era5_pl.grib: downloading 36/71 MB (50%)
  [Worker 2] 2025_05_era5_pl.grib: downloading 71/71 MB (100%)
  [36/48] 2025_05_era5_pl.grib done
  [Worker 4] 2025_02_era5_pl.grib: downloading 18/71 MB (25%)
  [Worker 13] 2024_05_era5_pl.grib: downloading 36/71 MB (50%)
  [Worker 11] 2024_07_era5_pl.grib: downloading 54/71 MB (75%)
  [Worker 8] 2025_11_era5_pl.grib: downloading 18/71 MB (25%)
  [Worker 12] 2025_09_era5_pl.grib: downloading 18/71 MB (25%)
  [Worker 9] 2025_08_era5_pl.grib: downloading 36/71 MB (50%)
  [Worker 10] 2025_10_era5_pl.grib: downloading 18/71 MB (25%)
  [Worker 7] 2025_07_era5_pl.grib: downloading 71/71 MB (100%)
  [37/48] 2025_07_era5_pl.grib done
  [Worker 0] 2025_12_era5_pl.grib: downloading 18/71 MB (25%)
  [Worker 9] 2025_08_era5_pl.grib: downloading 54/71 MB (75%)
  [Worker 4] 202

In [8]:
# PRINT RESULTS
for r in results:
    if r.success:
        print(f"Downloaded {r.task.target}")
    else:
        print(f"Failed {r.task.target}: {r.error}")

Downloaded output/2024_01_era5_sfc.grib
Downloaded output/2025_05_era5_sfc.grib
Downloaded output/2024_09_era5_sfc.grib
Downloaded output/2025_06_era5_sfc.grib
Downloaded output/2025_04_era5_sfc.grib
Downloaded output/2024_08_era5_sfc.grib
Downloaded output/2025_07_era5_sfc.grib
Downloaded output/2024_12_era5_sfc.grib
Downloaded output/2025_08_era5_sfc.grib
Downloaded output/2025_10_era5_sfc.grib
Downloaded output/2024_03_era5_sfc.grib
Downloaded output/2025_09_era5_sfc.grib
Downloaded output/2025_02_era5_sfc.grib
Downloaded output/2024_10_era5_sfc.grib
Downloaded output/2025_12_era5_sfc.grib
Downloaded output/2025_01_era5_sfc.grib
Downloaded output/2024_04_era5_sfc.grib
Downloaded output/2024_02_era5_sfc.grib
Downloaded output/2024_06_era5_sfc.grib
Downloaded output/2024_07_era5_sfc.grib
Downloaded output/2024_11_era5_sfc.grib
Downloaded output/2024_05_era5_sfc.grib
Downloaded output/2024_01_era5_pl.grib
Downloaded output/2024_04_era5_pl.grib
Downloaded output/2024_03_era5_pl.grib
Dow

In [ ]:
# TEST SOME FILES:
! grib_ls output/2024_01_era5_sfc.grib
! grib_ls output/2024_01_era5_pl.grib

output/2024_01_era5_sfc.grib
edition      centre       typeOfLevel  level        dataDate     stepRange    dataType     shortName    packingType  gridType     
1            ecmf         surface      0            20240101     0            an           2t           grid_simple  regular_ll  
1            ecmf         surface      0            20240101     0            an           10u          grid_simple  regular_ll  
1            ecmf         surface      0            20240101     0            an           10v          grid_simple  regular_ll  
1            ecmf         surface      0            20240101     5-6          fc           tp           grid_simple  regular_ll  
1            ecmf         surface      0            20240102     0            an           2t           grid_simple  regular_ll  
1            ecmf         surface      0            20240102     0            an           10u          grid_simple  regular_ll  
1            ecmf         surface      0            20240102